In [ ]:
# CNN CLASSIFICATION EXAMPLE — Using Our Data Format
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
# Load signal and background files
sig_df = pd.read_csv('Herwig.csv', header=None)
bkg_df = pd.read_csv('Jewel.csv', header=None)

In [ ]:
# THE DATA STRUCTURE
# Column 0:   Number of objects (particles) in this event
# Columns 1-4: Event-level features
# Columns 5-7: More event-level features
# Columns 8+:  Groups of 4 → (pT, eta, phi, flag) per object
# Trailing zeros: padding for shorter events

MAX_OBJECTS = 50           # Maximum number of objects to consider
N_FEATURES_PER_OBJ = 4    # Each object has 4 features: pT, eta, phi, flag
EVENT_FEATURES_START = 1   # Start of event-level features
EVENT_FEATURES_END = 8     # End of event-level features (exclusive)
OBJ_START_COL = 8          # Where per-object features begin

In [ ]:
# PARSE INTO STRUCTURED ARRAYS

def parse_events(df, max_objects=MAX_OBJECTS):
    """
    Parse raw CSV rows into:
      - event_features: (n_events, 7) — global event variables
      - object_array:   (n_events, max_objects, 4) — per-object features for CNN
    """
    n_events = len(df)
    raw = df.values                                    # Convert to numpy

    # Extract event-level features (columns 1 through 7)
    event_features = raw[:, EVENT_FEATURES_START:EVENT_FEATURES_END]

    # Extract per-object features into 3D array: (events, objects, features)
    object_array = np.zeros((n_events, max_objects, N_FEATURES_PER_OBJ))

    for i in range(n_events):
        n_obj = int(raw[i, 0])                         # Number of objects in this event
        n_obj = min(n_obj, max_objects)                 # Cap at max_objects

        for j in range(n_obj):
            col_start = OBJ_START_COL + j * N_FEATURES_PER_OBJ
            col_end = col_start + N_FEATURES_PER_OBJ

            # Check we don't exceed available columns
            if col_end <= raw.shape[1]:
                object_array[i, j, :] = raw[i, col_start:col_end]

    return event_features, object_array


In [ ]:
# Parse signal and background
sig_evt_feats, sig_obj_array = parse_events(sig_df)   # Signal
bkg_evt_feats, bkg_obj_array = parse_events(bkg_df)   # Background

print(f"Signal: {sig_obj_array.shape[0]} events, Object array shape: {sig_obj_array.shape}")
print(f"Background: {bkg_obj_array.shape[0]} events, Object array shape: {bkg_obj_array.shape}")

In [ ]:
# COMBINE AND LABEL

# Concatenate signal (label=1) and background (label=0)
X_objects = np.concatenate([sig_obj_array, bkg_obj_array], axis=0)    # (N, 50, 4)
X_event = np.concatenate([sig_evt_feats, bkg_evt_feats], axis=0)      # (N, 7)
y = np.concatenate([
    np.ones(len(sig_obj_array)),               # Signal labels
    np.zeros(len(bkg_obj_array))               # Background labels
])

print(f"\nCombined object array: {X_objects.shape}")  # (N, max_objects, 4)
print(f"Combined event features: {X_event.shape}")    # (N, 7)
print(f"Labels: {y.shape}, signal={y.sum():.0f}, background={len(y)-y.sum():.0f}")


In [ ]:
# TRAIN/TEST SPLIT

(X_obj_train, X_obj_test,
 X_evt_train, X_evt_test,
 y_train, y_test) = train_test_split(
    X_objects, X_event, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# SCALE EVENT-LEVEL FEATURES
# Note: For the CNN input (per-object), we scale per-feature across all objects

scaler_evt = StandardScaler()
X_evt_train_scaled = scaler_evt.fit_transform(X_evt_train)
X_evt_test_scaled = scaler_evt.transform(X_evt_test)

# Scale per-object features (reshape to 2D, scale, reshape back)
n_train = X_obj_train.shape[0]
n_test = X_obj_test.shape[0]

# Flatten objects: (N, 50, 4) → (N*50, 4), scale, reshape back
scaler_obj = StandardScaler()
X_obj_train_flat = X_obj_train.reshape(-1, N_FEATURES_PER_OBJ)     # (N*50, 4)
X_obj_test_flat = X_obj_test.reshape(-1, N_FEATURES_PER_OBJ)       # (N*50, 4)

X_obj_train_scaled = scaler_obj.fit_transform(X_obj_train_flat).reshape(n_train, MAX_OBJECTS, N_FEATURES_PER_OBJ)
X_obj_test_scaled = scaler_obj.transform(X_obj_test_flat).reshape(n_test, MAX_OBJECTS, N_FEATURES_PER_OBJ)

print(f"\nTrain objects shape: {X_obj_train_scaled.shape}")  # (N_train, 50, 4)
print(f"Test objects shape:  {X_obj_test_scaled.shape}")


In [ ]:
# BUILD CNN MODEL
# Architecture: 1D CNN over the "objects" axis with 4 feature channels
# Then merge with event-level features in a dense head

def build_cnn_classifier(n_objects, n_obj_features, n_event_features):
    """
    Build a 1D CNN that processes per-object features and combines
    with event-level features for binary classification.

    Input shape for CNN: (n_objects, n_obj_features) = (50, 4)
    Think of it as: 50 "time steps" with 4 "channels"
    """

    # --- Object branch (CNN) ---
    obj_input = layers.Input(shape=(n_objects, n_obj_features), name='object_input')

    # First conv block: learn local patterns across neighboring objects
    x = layers.Conv1D(32, kernel_size=3, padding='same')(obj_input)   # 32 filters
    x = layers.BatchNormalization()(x)                                 # Normalize
    x = layers.ReLU()(x)                                               # Activation
    x = layers.MaxPooling1D(pool_size=2)(x)                           # Downsample: 50→25

    # Second conv block: learn higher-level patterns
    x = layers.Conv1D(64, kernel_size=3, padding='same')(x)           # 64 filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)                           # Downsample: 25→12

    # Third conv block
    x = layers.Conv1D(128, kernel_size=3, padding='same')(x)          # 128 filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)                            # Pool: 12→1 (128-dim)

    # --- Event-level branch (Dense) ---
    evt_input = layers.Input(shape=(n_event_features,), name='event_input')
    e = layers.Dense(32, activation='relu')(evt_input)                 # Small dense layer
    e = layers.BatchNormalization()(e)

    # --- Merge both branches ---
    merged = layers.Concatenate()([x, e])                             # (128 + 32) = 160

    # Classification head
    merged = layers.Dense(64, activation='relu')(merged)               # Dense layer
    merged = layers.Dropout(0.3)(merged)                              # Regularization
    merged = layers.Dense(32, activation='relu')(merged)               # Another dense
    merged = layers.Dropout(0.2)(merged)
    output = layers.Dense(1, activation='sigmoid')(merged)             # Binary output

    model = keras.Model(inputs=[obj_input, evt_input], outputs=output)
    return model


In [ ]:

# Build
cnn_model = build_cnn_classifier(
    n_objects=MAX_OBJECTS,
    n_obj_features=N_FEATURES_PER_OBJ,
    n_event_features=X_evt_train_scaled.shape[1]
)

# Compile
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Print summary
cnn_model.summary()


In [ ]:
# TRAIN THE MODEL

history = cnn_model.fit(
    [X_obj_train_scaled, X_evt_train_scaled],       # Two inputs
    y_train,                                         # Labels
    epochs=50,                                       # Max epochs
    batch_size=64,                                   # Batch size
    validation_split=0.15,                           # 15% for validation
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True                # Keep best model
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5
        )
    ],
    verbose=1
)


In [ ]:
# EVALUATE

# Predict probabilities
y_prob = cnn_model.predict(
    [X_obj_test_scaled, X_evt_test_scaled], verbose=0
).ravel()

# Hard predictions at threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nCNN Classification Results:")
print(f"  Accuracy: {acc:.4f}")
print(f"  AUC-ROC:  {auc:.4f}")


In [ ]:
# PLOT TRAINING CURVES

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Binary Cross-Entropy Loss'); ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Validation')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Classification Accuracy'); ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
# PLOT ROC CURVE

from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'CNN (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — CNN Classifier')
plt.legend(); plt.tight_layout(); plt.show()